# Fine-tuning for RQ3 (LoRA): one adapter per model size

Runs on a free Google Colab or Kaggle GPU (T4, 15 GB). **Runtime > Change runtime type > T4 GPU.**

What it does:
1. trains a LoRA adapter (QLoRA: the base model is loaded in 4 bits) on `lora_train.jsonl`, the chats built from the 440 Detroit training situations by `prepare_lora_data.py`. No test situation is used;
2. reports the loss on the held-out `lora_val.jsonl` (44 training situations kept aside);
3. converts the adapter to GGUF so that Ollama can load it on top of the same 4-bit `qwen2.5` model used everywhere else.

Set `SIZE` below and run all cells once per size (0.5B, 1.5B, 3B). Download the `.gguf` file at the end, then follow `README.md`.

Status: written and checked for consistency, **not yet run** (no GPU was available where it was written).

In [ ]:
SIZE = '1.5B'          # '0.5B', '1.5B' or '3B'
EPOCHS = 2
LR = 2e-4
RANK = 16
BASE = f'Qwen/Qwen2.5-{SIZE}-Instruct'
OUT = f'qwen2.5-{SIZE.lower()}-ev-lora'

In [ ]:
!pip -q install transformers==4.46.3 peft==0.13.2 trl==0.12.1 datasets==3.1.0 accelerate==1.1.1 bitsandbytes==0.44.1

Upload `lora_train.jsonl` and `lora_val.jsonl` (from the `lora` folder) with the file panel on the left, or run the next cell to pick them.

In [ ]:
import os
if not (os.path.exists('lora_train.jsonl') and os.path.exists('lora_val.jsonl')):
    from google.colab import files
    files.upload()
print(sum(1 for _ in open('lora_train.jsonl')), 'training chats;', sum(1 for _ in open('lora_val.jsonl')), 'validation chats')

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

torch.manual_seed(0)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, device_map='auto', torch_dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16))
data = load_dataset('json', data_files={'train': 'lora_train.jsonl', 'val': 'lora_val.jsonl'})
fmt = lambda ex: {'text': tok.apply_chat_template(ex['messages'], tokenize=False)}
data = data.map(fmt, remove_columns=['messages'])
print(data['train'][0]['text'][-400:])

Only the assistant's answer is trained on (the system prompt, fact sheet and request are masked), so the model learns to answer, not to repeat the question.

In [ ]:
collator = DataCollatorForCompletionOnlyLM(response_template='<|im_start|>assistant\n', tokenizer=tok)
peft_cfg = LoraConfig(r=RANK, lora_alpha=2 * RANK, lora_dropout=0.05, task_type='CAUSAL_LM',
                      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
args = SFTConfig(output_dir=OUT, num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=4, gradient_accumulation_steps=4,
                 per_device_eval_batch_size=4, eval_strategy='epoch', save_strategy='no', logging_steps=10, lr_scheduler_type='cosine',
                 warmup_ratio=0.05, fp16=True, max_seq_length=2048, dataset_text_field='text', seed=0, report_to='none')
trainer = SFTTrainer(model=model, args=args, train_dataset=data['train'], eval_dataset=data['val'], peft_config=peft_cfg,
                     data_collator=collator, tokenizer=tok)
trainer.train()
print(trainer.evaluate())
trainer.model.save_pretrained(OUT); tok.save_pretrained(OUT)

Write down the validation loss printed above for each size; it goes into the thesis next to the adapter settings.

Convert the adapter to GGUF for Ollama (needs the base model files and llama.cpp's converter):

In [ ]:
!git clone -q --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
from huggingface_hub import snapshot_download
base_dir = snapshot_download(BASE)
!python llama.cpp/convert_lora_to_gguf.py {OUT} --base {base_dir} --outtype f16 --outfile {OUT}.gguf
print(os.path.getsize(f'{OUT}.gguf') / 1e6, 'MB')

In [ ]:
from google.colab import files
files.download(f'{OUT}.gguf')